# Week 8: Predator-Prey Dynamics

## SCIE1500 — Analytical Methods for Scientists
### Theme: "Ecological Interactions"

---

## Overview

**Science Context:** Modeling interacting species, population cycles, ecological stability

**Learning Outcomes:** By the end of this week you should be able to:
1. Understand coupled differential equations for interacting populations
2. Write and interpret the Lotka-Volterra predator-prey model
3. Find and analyze equilibrium points
4. Interpret phase portraits and trajectories
5. Understand the oscillatory behavior of predator-prey systems

**Exam Alignment:** Q19, Q20

## Setup: Import Required Libraries

Run this cell first to import all the Python libraries we'll use throughout this lesson.

In [ ]:
# Standard imports for SCIE1500 notebooks
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import odeint

# Set plotting style
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

print("✓ Libraries loaded successfully!")

---

## 1. Introduction: Why Model Predator-Prey Dynamics?

### Real-World Examples

Predator-prey interactions are everywhere:
- Lynx and snowshoe hares in Canada
- Wolves and moose on Isle Royale
- Sharks and fish in marine ecosystems
- Ladybugs and aphids in gardens

### The Key Observation

These populations often exhibit **oscillations**: when prey is abundant, predators thrive and increase; but then predators reduce prey, leading to predator decline, allowing prey to recover.

This cyclic behavior emerges naturally from mathematical models.

In [ ]:
# Historical data: Lynx and Hare populations (Hudson Bay Company)

years = np.array([1900, 1905, 1910, 1915, 1920, 1925, 1930, 1935])
hare = np.array([30, 50, 100, 60, 25, 45, 90, 55])  # Thousands
lynx = np.array([5, 15, 40, 55, 30, 10, 25, 50])    # Thousands

plt.figure(figsize=(12, 5))
plt.plot(years, hare, 'b-o', linewidth=2, markersize=8, label='Hare (Prey)')
plt.plot(years, lynx, 'r-s', linewidth=2, markersize=8, label='Lynx (Predator)')

plt.xlabel('Year', fontsize=12)
plt.ylabel('Population (thousands)', fontsize=12)
plt.title('Classic Predator-Prey Oscillations: Lynx and Hare', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Notice: Predator peaks FOLLOW prey peaks (with a time lag).")

---

## 2. The Lotka-Volterra Model

### The Equations

Let:
- $N(t)$ = prey population (e.g., rabbits)
- $P(t)$ = predator population (e.g., foxes)

The **Lotka-Volterra equations** are:

$$\frac{dN}{dt} = rN - aNP$$

$$\frac{dP}{dt} = -mP + bNP$$

### Parameter Meanings

| Parameter | Meaning | Effect |
|-----------|---------|--------|
| $r$ | Prey intrinsic growth rate | Prey grows exponentially without predators |
| $a$ | Predation rate | Rate at which predators consume prey |
| $m$ | Predator mortality rate | Predators die without prey |
| $b$ | Predator efficiency | How much predators benefit from eating prey |

### Understanding the Terms

**Prey equation:** $\frac{dN}{dt} = rN - aNP$
- $rN$: Exponential growth (births exceed deaths)
- $-aNP$: Deaths due to predation (depends on encounters, hence $N \times P$)

**Predator equation:** $\frac{dP}{dt} = -mP + bNP$
- $-mP$: Exponential decay (deaths without food)
- $+bNP$: Births from consuming prey (more prey → more predator births)

In [ ]:
# Lotka-Volterra model implementation

def lotka_volterra(y, t, r, a, m, b):
    """
    Lotka-Volterra predator-prey equations.
    y = [N, P] where N = prey, P = predator
    """
    N, P = y
    dNdt = r*N - a*N*P      # Prey dynamics
    dPdt = -m*P + b*N*P     # Predator dynamics
    return [dNdt, dPdt]

# Parameters
r = 1.0    # Prey growth rate
a = 0.1    # Predation rate
m = 1.5    # Predator mortality
b = 0.075  # Predator efficiency

# Initial conditions
N0 = 40    # Initial prey
P0 = 9     # Initial predators

# Time array
t = np.linspace(0, 50, 1000)

# Solve the ODEs
solution = odeint(lotka_volterra, [N0, P0], t, args=(r, a, m, b))
N = solution[:, 0]  # Prey
P = solution[:, 1]  # Predator

# Plot
plt.figure(figsize=(12, 5))
plt.plot(t, N, 'b-', linewidth=2, label='Prey N(t)')
plt.plot(t, P, 'r-', linewidth=2, label='Predator P(t)')

plt.xlabel('Time', fontsize=12)
plt.ylabel('Population', fontsize=12)
plt.title('Lotka-Volterra Predator-Prey Model', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Parameters: r = {r}, a = {a}, m = {m}, b = {b}")
print(f"Initial conditions: N₀ = {N0}, P₀ = {P0}")
print("\nNote the periodic oscillations - both populations cycle!")

---

## 3. Equilibrium Points

### Finding Equilibria

Equilibrium occurs when both populations are unchanging:

$$\frac{dN}{dt} = 0 \quad \text{and} \quad \frac{dP}{dt} = 0$$

### Solving for Equilibrium

From $\frac{dN}{dt} = rN - aNP = N(r - aP) = 0$:
- Either $N = 0$ or $P = r/a$

From $\frac{dP}{dt} = -mP + bNP = P(-m + bN) = 0$:
- Either $P = 0$ or $N = m/b$

**Two equilibrium points:**
1. **Trivial:** $(N^*, P^*) = (0, 0)$ — extinction
2. **Coexistence:** $(N^*, P^*) = (m/b, r/a)$ — both species persist

In [ ]:
# Calculate and visualize equilibrium

# Equilibrium point (coexistence)
N_star = m / b
P_star = r / a

print("Equilibrium Analysis")
print("="*40)
print(f"Parameters: r = {r}, a = {a}, m = {m}, b = {b}")
print()
print(f"Trivial equilibrium: (0, 0)")
print(f"Coexistence equilibrium: (N*, P*) = ({N_star:.1f}, {P_star:.1f})")
print()
print(f"  N* = m/b = {m}/{b} = {N_star:.1f}")
print(f"  P* = r/a = {r}/{a} = {P_star:.1f}")

# Verify: at equilibrium, derivatives should be zero
dN_eq = r*N_star - a*N_star*P_star
dP_eq = -m*P_star + b*N_star*P_star
print(f"\nVerification at equilibrium:")
print(f"  dN/dt = {dN_eq:.6f} (should be 0)")
print(f"  dP/dt = {dP_eq:.6f} (should be 0)")

---

## 4. Phase Portrait Analysis

### What is a Phase Portrait?

A **phase portrait** plots $P$ vs $N$ (predators vs prey), eliminating time. Each point represents a state of the system, and trajectories show how the system evolves.

### Isoclines

**Isoclines** are curves where one variable has zero rate of change:

- **Prey isocline** ($dN/dt = 0$): $P = r/a$ (horizontal line)
- **Predator isocline** ($dP/dt = 0$): $N = m/b$ (vertical line)

In [ ]:
# Phase portrait

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Phase portrait with trajectory
ax1.plot(N, P, 'b-', linewidth=2, label='Trajectory')
ax1.scatter([N0], [P0], color='green', s=100, zorder=5, label='Start')
ax1.scatter([N_star], [P_star], color='red', s=150, marker='*', zorder=5, label=f'Equilibrium ({N_star:.0f}, {P_star:.0f})')

# Isoclines
ax1.axhline(y=P_star, color='orange', linestyle='--', linewidth=2, alpha=0.7, label=f'Prey isocline (P = {P_star:.0f})')
ax1.axvline(x=N_star, color='purple', linestyle='--', linewidth=2, alpha=0.7, label=f'Predator isocline (N = {N_star:.0f})')

# Direction arrows
arrow_scale = 1.5
regions = [(10, 5), (30, 5), (30, 15), (10, 15)]
for n, p in regions:
    dn = (r*n - a*n*p) * arrow_scale
    dp = (-m*p + b*n*p) * arrow_scale
    ax1.arrow(n, p, dn*2, dp*2, head_width=1, head_length=0.5, fc='gray', ec='gray')

ax1.set_xlabel('Prey N', fontsize=12)
ax1.set_ylabel('Predator P', fontsize=12)
ax1.set_title('Phase Portrait', fontsize=14)
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, max(N)*1.1)
ax1.set_ylim(0, max(P)*1.1)

# Right: Multiple trajectories from different starting points
initial_conditions = [(40, 9), (30, 5), (50, 15), (20, 12)]
colors = ['blue', 'green', 'orange', 'purple']

for (n0, p0), color in zip(initial_conditions, colors):
    sol = odeint(lotka_volterra, [n0, p0], t, args=(r, a, m, b))
    ax2.plot(sol[:, 0], sol[:, 1], color=color, linewidth=1.5, label=f'Start ({n0}, {p0})')
    ax2.scatter([n0], [p0], color=color, s=50, zorder=5)

ax2.scatter([N_star], [P_star], color='red', s=150, marker='*', zorder=5, label='Equilibrium')
ax2.axhline(y=P_star, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax2.axvline(x=N_star, color='gray', linestyle='--', linewidth=1, alpha=0.5)

ax2.set_xlabel('Prey N', fontsize=12)
ax2.set_ylabel('Predator P', fontsize=12)
ax2.set_title('Multiple Trajectories (Closed Orbits)', fontsize=14)
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observation: Trajectories form CLOSED ORBITS around the equilibrium.")
print("The equilibrium is a CENTER - neutrally stable.")

---

## 5. Understanding the Quadrants

The isoclines divide the phase plane into four regions:

| Region | N vs N* | P vs P* | N trend | P trend |
|--------|---------|---------|---------|--------|
| I | N < N* | P < P* | ↑ (growing) | ↓ (declining) |
| II | N > N* | P < P* | ↑ (growing) | ↑ (growing) |
| III | N > N* | P > P* | ↓ (declining) | ↑ (growing) |
| IV | N < N* | P > P* | ↓ (declining) | ↓ (declining) |

In [ ]:
# Visualize the four regions and their dynamics

fig, ax = plt.subplots(figsize=(10, 8))

# Draw isoclines
ax.axhline(y=P_star, color='blue', linestyle='-', linewidth=3, label=f'dN/dt = 0 (P = {P_star:.0f})')
ax.axvline(x=N_star, color='red', linestyle='-', linewidth=3, label=f'dP/dt = 0 (N = {N_star:.0f})')
ax.scatter([N_star], [P_star], color='black', s=200, marker='o', zorder=5, label='Equilibrium')

# Shade regions
ax.fill([0, N_star, N_star, 0], [0, 0, P_star, P_star], alpha=0.2, color='green')
ax.fill([N_star, 50, 50, N_star], [0, 0, P_star, P_star], alpha=0.2, color='yellow')
ax.fill([N_star, 50, 50, N_star], [P_star, P_star, 20, 20], alpha=0.2, color='orange')
ax.fill([0, N_star, N_star, 0], [P_star, P_star, 20, 20], alpha=0.2, color='red')

# Label regions
ax.text(N_star/2, P_star/2, 'Region I\nN↑ P↓', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(N_star + (50-N_star)/2, P_star/2, 'Region II\nN↑ P↑', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(N_star + (50-N_star)/2, P_star + (20-P_star)/2, 'Region III\nN↓ P↑', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(N_star/2, P_star + (20-P_star)/2, 'Region IV\nN↓ P↓', ha='center', va='center', fontsize=12, fontweight='bold')

# Draw arrows showing direction of movement
arrow_positions = [(8, 4), (30, 4), (30, 14), (8, 14)]
for n, p in arrow_positions:
    dn = r*n - a*n*p
    dp = -m*p + b*n*p
    ax.annotate('', xy=(n + dn*3, p + dp*3), xytext=(n, p),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))

ax.set_xlabel('Prey N', fontsize=12)
ax.set_ylabel('Predator P', fontsize=12)
ax.set_title('Phase Plane Dynamics: Four Regions', fontsize=14)
ax.legend(loc='upper right')
ax.set_xlim(0, 50)
ax.set_ylim(0, 20)
ax.grid(True, alpha=0.3)
plt.show()

print("The arrows show the direction of movement in each region.")
print("Together, they create the counter-clockwise cycling behavior.")

---

## 6. Conservation Law

### The Lotka-Volterra Conserved Quantity

The Lotka-Volterra system has a remarkable property: there exists a quantity that remains constant along trajectories:

$$H(N, P) = bN - m\ln(N) + aP - r\ln(P) = \text{constant}$$

This explains why the orbits are closed: the system never spirals in or out.

In [ ]:
# Verify the conserved quantity

def H(N, P, r, a, m, b):
    """Lotka-Volterra conserved quantity (Hamiltonian)"""
    return b*N - m*np.log(N) + a*P - r*np.log(P)

# Calculate H along the trajectory
H_values = H(N, P, r, a, m, b)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: H over time
ax1.plot(t, H_values, 'b-', linewidth=2)
ax1.axhline(y=H_values[0], color='red', linestyle='--', linewidth=2, label=f'H = {H_values[0]:.4f}')
ax1.set_xlabel('Time', fontsize=12)
ax1.set_ylabel('H(N, P)', fontsize=12)
ax1.set_title('Conserved Quantity H(N, P) Over Time', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Contours of H in phase space
N_grid = np.linspace(1, 60, 100)
P_grid = np.linspace(1, 20, 100)
N_mesh, P_mesh = np.meshgrid(N_grid, P_grid)
H_mesh = H(N_mesh, P_mesh, r, a, m, b)

contours = ax2.contour(N_mesh, P_mesh, H_mesh, levels=15, cmap='viridis')
ax2.clabel(contours, inline=True, fontsize=8)
ax2.scatter([N_star], [P_star], color='red', s=150, marker='*', zorder=5)
ax2.plot(N, P, 'r-', linewidth=2, label='Trajectory')

ax2.set_xlabel('Prey N', fontsize=12)
ax2.set_ylabel('Predator P', fontsize=12)
ax2.set_title('Level Curves of H (Trajectories!)', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Conservation verified: H remains constant at {H_values[0]:.4f}")
print(f"Max deviation: {abs(H_values - H_values[0]).max():.6f} (numerical error)")

---

## 7. Average Populations

### A Surprising Result

Over one complete cycle, the **time-averaged populations** equal the equilibrium values:

$$\bar{N} = N^* = \frac{m}{b}$$
$$\bar{P} = P^* = \frac{r}{a}$$

In [ ]:
# Calculate average populations

# Find period (approximate by finding peaks)
from scipy.signal import find_peaks
peaks, _ = find_peaks(N)

if len(peaks) >= 2:
    period_idx = peaks[1] - peaks[0]
    period = t[peaks[1]] - t[peaks[0]]
    
    # Average over one period
    N_avg = np.mean(N[peaks[0]:peaks[1]])
    P_avg = np.mean(P[peaks[0]:peaks[1]])
    
    print("Average Populations Over One Cycle")
    print("="*40)
    print(f"Estimated period: T ≈ {period:.2f}")
    print()
    print(f"Average prey: N̄ = {N_avg:.2f}")
    print(f"Equilibrium N*: = {N_star:.2f}")
    print()
    print(f"Average predator: P̄ = {P_avg:.2f}")
    print(f"Equilibrium P*: = {P_star:.2f}")
    print()
    print("The averages match the equilibrium values! (approximately)")

---

## 8. Effect of Parameters

What happens when we change the model parameters?

In [ ]:
# Explore parameter effects

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Base parameters
base_params = {'r': 1.0, 'a': 0.1, 'm': 1.5, 'b': 0.075}

scenarios = [
    ('Increased prey growth (r)', {'r': 1.5}),
    ('Increased predation rate (a)', {'a': 0.15}),
    ('Increased predator mortality (m)', {'m': 2.0}),
    ('Increased predator efficiency (b)', {'b': 0.1})
]

for ax, (title, changes) in zip(axes.flatten(), scenarios):
    params = base_params.copy()
    params.update(changes)
    
    sol = odeint(lotka_volterra, [N0, P0], t, args=(params['r'], params['a'], params['m'], params['b']))
    
    # Phase plot
    ax.plot(sol[:, 0], sol[:, 1], 'b-', linewidth=1.5)
    
    # Equilibrium
    N_eq = params['m'] / params['b']
    P_eq = params['r'] / params['a']
    ax.scatter([N_eq], [P_eq], color='red', s=100, marker='*', zorder=5)
    ax.scatter([N0], [P0], color='green', s=50, zorder=5)
    
    ax.set_xlabel('Prey N')
    ax.set_ylabel('Predator P')
    ax.set_title(f'{title}\nEquilibrium: ({N_eq:.1f}, {P_eq:.1f})')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Parameter effects on equilibrium:")
print("  ↑r (prey growth) → ↑P* (more predators)")
print("  ↑a (predation) → ↓P* (fewer predators)")
print("  ↑m (mortality) → ↑N* (more prey)")
print("  ↑b (efficiency) → ↓N* (fewer prey)")

---

## Key Formulas Summary

| Topic | Key Formula |
|-------|-------------|
| Prey equation | $\frac{dN}{dt} = rN - aNP$ |
| Predator equation | $\frac{dP}{dt} = -mP + bNP$ |
| Prey equilibrium | $N^* = m/b$ |
| Predator equilibrium | $P^* = r/a$ |
| Prey isocline | $P = r/a$ |
| Predator isocline | $N = m/b$ |

---

## Self-Assessment Checklist

Before moving to Week 9, ensure you can:

- [ ] Write the Lotka-Volterra equations and explain each term
- [ ] Find equilibrium points by setting derivatives to zero
- [ ] Draw and interpret a phase portrait
- [ ] Explain why predator peaks lag behind prey peaks
- [ ] Describe the behavior in each of the four regions
- [ ] Explain why orbits are closed in the basic model

---

*Next: Week 9 — Probability Foundations*